# Otros optimizadores: PSO y QNG

[`06a`](06a_entrenamiento_variacional.ipynb) entrenó un QAOA con Evolución Diferencial. Polypus trae otros dos optimizadores para el mismo `train()`: PSO, que apenas cambia la llamada, y QNG, que necesita una pieza extra propia del circuito. Este notebook entrena el mismo problema de MaxCut con los dos.

## El mismo circuito

Se reutiliza el QAOA de una capa sobre el mismo grafo camino de tres nodos de [`06a`](06a_entrenamiento_variacional.ipynb), con el mejor corte posible ya calculado allí por fuerza bruta:

In [ ]:
import polypus

Param = polypus.Param
edges = [(0, 1), (1, 2)]


def cut_value(bitstring):
    bits = bitstring[::-1]
    return sum(1 for i, j in edges if bits[i] != bits[j])


qc = polypus.Circuit(3)
qc.h(0).h(1).h(2)
qc.rzz(0, 1, Param(0))
qc.rzz(1, 2, Param(0))
qc.rx(0, Param(1)).rx(1, Param(1)).rx(2, Param(1))
qc.measure_all()

mejor_corte = 2

## PSO: la misma API, cambia el optimizador

Particle Swarm Optimization, PSO, mantiene un enjambre de partículas que se mueven por el espacio de parámetros, atraídas por su propio mejor resultado y por el mejor de todo el enjambre. La llamada a `train()` es idéntica a la de [`06a`](06a_entrenamiento_variacional.ipynb): mismo circuito, mismos argumentos de infraestructura, mismo `expectation_function`. Lo único que cambia es `polypus.DE(...)` por `polypus.PSO(...)`:

In [ ]:
pso = polypus.PSO(generations=60, population_size=20, tolerance=0.001, seed=7)

result_pso = polypus.train(
    qc,
    pso,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-pso",
)
print(result_pso.best_params)
print(result_pso.best_fitness)
print(result_pso.iterations_run)
print(result_pso.converged)

`iterations_run` llega a las 60 completas y `converged` sale `False`: a diferencia de la Evolución Diferencial de [`06a`](06a_entrenamiento_variacional.ipynb), que paró antes de tiempo al estabilizarse, PSO agota todo el presupuesto sin bajar de `tolerance`, aunque el `best_fitness` encontrado es prácticamente el mismo. Evaluado igual que en [`06a`](06a_entrenamiento_variacional.ipynb), bindeando `best_params` y midiendo:

In [ ]:
qc_final_pso = polypus.Circuit.from_qasm2(qc.to_qasm2(result_pso.best_params))
resultado_pso = polypus.run_quantum_circuit(qc_final_pso, shots=2000, infrastructure="local")
counts_pso = resultado_pso.counts[0]
total_pso = sum(counts_pso.values())
corte_medio_pso = sum(cut_value(bits) * n for bits, n in counts_pso.items()) / total_pso

print("corte medio medido:", corte_medio_pso)
print("mejor corte posible:", mejor_corte)

Un resultado prácticamente igual al de PSO.

## QNG: el mismo circuito, una pieza extra

Quantum Natural Gradient, QNG, también sigue el gradiente de la función de coste, pero corrigiéndolo con la geometría propia del espacio de parámetros del circuito, en vez de tratarlo como un espacio plano. A cambio de esa corrección, pide algo que DE y PSO no piden: `variance_function`, una función que estima cuánto varía, en el punto actual, el resultado de medir el generador de cada parámetro. Polypus no puede calcularla sola porque depende de la estructura concreta de cada circuito.

Para el parámetro `beta`, índice 1, ese generador es la suma de `X` en cada qubit, la misma rotación que aplica la capa de mezcla. Su varianza se estima construyendo el circuito hasta justo antes de esa capa, con `gamma` ya fijado al valor actual, rotando a la base `X` con una H antes de medir, y calculando la varianza de la suma de los signos medidos. Para el parámetro `gamma`, índice 0, no hay ninguna capa previa que fijar: por convención, su varianza se toma como 0, y la regularización de `tikhonov_reg` se encarga de que eso no cause una división por un número demasiado pequeño:

In [ ]:
def variance_fn(theta, a):
    if a == 0:
        return 0.0

    qc_parcial = polypus.Circuit(3)
    qc_parcial.h(0).h(1).h(2)
    qc_parcial.rzz(0, 1, theta[0])
    qc_parcial.rzz(1, 2, theta[0])
    qc_parcial.h(0).h(1).h(2)  # rota a la base X antes de medir
    qc_parcial.measure_all()

    # seed fija: el train() exterior ya tiene su propia seed, pero esta
    # llamada es independiente y sin ella el resultado no sería reproducible
    resultado = polypus.run_quantum_circuit(
        qc_parcial, shots=2000, infrastructure="local", seed=7
    )
    counts = resultado.counts[0]
    total = sum(counts.values())

    exp_h = 0.0
    exp_h2 = 0.0
    for bitstring, n in counts.items():
        prob = n / total
        bits = bitstring[::-1]
        valor = sum(1 if bits[i] == "0" else -1 for i in range(3))
        exp_h += valor * prob
        exp_h2 += (valor**2) * prob

    return exp_h2 - exp_h**2

`polypus.QNG` pide `variance_function` como primer argumento, la pieza nueva que no tienen ni `DE` ni `PSO`. A cambio, no tiene ningún parámetro `tolerance`: QNG no tiene un criterio de parada temprana, así que `converged` sale siempre `False`, haya mejorado o no en las últimas iteraciones. El resto de `train()` no cambia:

In [ ]:
qng = polypus.QNG(
    variance_fn,
    max_iters=40,
    learning_rate=0.1,
    finite_difference_step=0.1,
    tikhonov_reg=0.05,
    seed=7,
)

result_qng = polypus.train(
    qc,
    qng,
    shots=1000,
    n_qpus=1,
    dimensions=qc.num_params,
    expectation_function=cut_value,
    infrastructure="local",
    nodes=1,
    cores_per_qpu=2,
    id="maxcut-qng",
)
print(result_qng.best_params)
print(result_qng.best_fitness)
print(result_qng.converged)

Evaluado igual que los dos anteriores:

In [ ]:
qc_final_qng = polypus.Circuit.from_qasm2(qc.to_qasm2(result_qng.best_params))
resultado_qng = polypus.run_quantum_circuit(qc_final_qng, shots=2000, infrastructure="local")
counts_qng = resultado_qng.counts[0]
total_qng = sum(counts_qng.values())
corte_medio_qng = sum(cut_value(bits) * n for bits, n in counts_qng.items()) / total_qng

print("corte medio medido:", corte_medio_qng)
print("mejor corte posible:", mejor_corte)

El corte medio de QNG queda algo por debajo del de DE y PSO, con esta configuración sin ajustar de `learning_rate` y `finite_difference_step`: el punto de este notebook es mostrar la API, no exprimir cada optimizador al máximo.

## Resumen

Este notebook ha entrenado el mismo QAOA de [`06a`](06a_entrenamiento_variacional.ipynb) con dos optimizadores más: PSO, con la misma llamada a `train()` salvo el objeto pasado en `method`, y QNG, que además pide una `variance_function` propia del circuito para corregir el gradiente con la geometría del espacio de parámetros.

## Fin de la serie

Esta serie ha recorrido el camino completo: desde el primer circuito hasta entrenar uno con datos. Quien quiera seguir puede consultar `examples/`, con casos más grandes, entre ellos una versión completa de MaxCut con QAOA, y el README principal, que documenta benchmarks de rendimiento entre backends.

Vuelve al índice: [`00_bienvenida.ipynb`](00_bienvenida.ipynb).